# Full Pipeline Demo

Demonstrates the closed-loop pipeline:

    base_prompt -> LLM -> evaluate -> if low, tune prompt -> LLM -> ...

The `llm` parameter is a callable `(prompt, sub_categories) -> List[str]`.
Below it is a **mock LLM** that returns different questions on each call,
simulating what a real LLM would do after seeing a refined prompt.

To use a real LLM:
1. Store an API key in `.env` (see `.env.example`)
2. Write a function that calls your LLM and returns a list of question strings
3. Pass it as the `llm` argument to `pipeline.run`

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline import run


# A mock LLM that returns different questions on each call, so the
# pipeline loop has something to observe. In a real run, replace this
# with a function that calls your LLM provider.
_ITERATIONS = {"call": 0}

_QUESTIONS_BY_CALL = [
    # Call 1 — base prompt output. Vague and repetitive on purpose.
    [
        "Is maintenance good in your plant?",
        "How is maintenance handled?",
        "Is there some kind of maintenance system?",
        "Are things working well with maintenance?",
    ],
    # Call 2 — first refinement. More specific but still narrow.
    [
        "Does your maintenance strategy include predictive analytics on critical assets?",
        "How frequently per month are critical assets inspected against a schedule?",
        "Is there a documented procedure for handling equipment anomalies?",
        "Does the CMMS log every failure with a timestamp and root cause?",
    ],
    # Call 3 — second refinement. Broader coverage of sub-categories.
    [
        "How many hours did the CNC machine run last week before failure?",
        "What percentage of maintenance issues are reported through the CMMS?",
        "Does a documented procedure exist for each type of maintenance task?",
        "How are maintenance resources (technicians, spare parts) allocated?",
        "How often per quarter is the maintenance plan reviewed and updated?",
        "Is every maintenance action logged with a timestamp and root cause?",
    ],
]


def mock_llm(prompt, sub_categories):
    _ITERATIONS["call"] += 1
    idx = min(_ITERATIONS["call"] - 1, len(_QUESTIONS_BY_CALL) - 1)
    return list(_QUESTIONS_BY_CALL[idx])

In [2]:
sub_categories = [
    "Identification", "Reporting", "Execution",
    "Resource Management", "Planning", "Documentation",
]

result = run(
    base_prompt="Generate questions for Industry 4.0 maintenance readiness assessment",
    llm=mock_llm,
    sub_categories=sub_categories,
    max_iterations=3,
    quality_threshold=0.85,
    verbose=True,
)

print()
print("Final prompt:")
print(result["final_prompt"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  iter 1: n=  4  spec=2.31  rep=0.68  cov=0.17  quality=0.272


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\gaura\OneDrive\Desktop\New Beginnings\assessai-toolkit\venv\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  iter 2: n=  4  spec=4.25  rep=0.28  cov=0.17  quality=0.566


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\gaura\OneDrive\Desktop\New Beginnings\assessai-toolkit\venv\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  iter 3: n=  6  spec=3.92  rep=0.41  cov=0.33  quality=0.551

Final prompt:
Generate questions for Industry 4.0 maintenance readiness assessment. Include questions covering: terms related to maintenance, handled, maintenance handled, kind maintenance, kind; terms related to maintenance, good, plant. Include questions covering: terms related to procedure, procedure handling, anomalies, documented, equipment anomalies; terms related to schedule, month critical, month, assets, frequently.


## What this demo shows and what it does not

**Shows:** the loop mechanically works. Given a mock that returns better
questions after each prompt refinement, composite quality increases and
the pipeline converges.

**Does not show:** that a real LLM would actually improve its questions
in response to our refined prompt. Live LLM validation was not run. The
`llm` argument is pluggable — anyone with an API key can test that
hypothesis.

This gap is documented in the README's limitations section.

**A note on the trade-off.** Composite quality does not always increase
monotonically. In the run above, iteration 3 added more questions, which
raised coverage (0.17 -> 0.33) but also lowered average specificity
(4.25 -> 3.92), so composite quality dipped slightly (0.566 -> 0.551).
This is a real trade-off in the metrics, not a bug. The user can weight
specificity, repetition, and coverage according to their priorities via
the `weights` argument to `composite_quality`.